In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT + MODEL
# ============================================================

import os

# ------------------------------------------------------------
# IMPORTANT:
# Start with ONE T4 first.
# This avoids PyTorch DataParallel complications while
# verifying the hybrid architecture.
# ------------------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import torch

from kaggle_secrets import UserSecretsClient

from huggingface_hub import get_token, whoami

from transformers import (
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    DonutProcessor,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from datasets import load_dataset
from torch.utils.data import Dataset


# ============================================================
# CONFIGURATION
# ============================================================

TOKENIZER_ID = "NLPC-UOM/SinBERT-large"

TROCR_HUB_ID = (
    "danush99/Model_TrOCR-Sin-Handwritten-Text"
)

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "danushamsc25/sinfunddonut/"
    "SinFundDonut"
)

TRAIN_PATH = os.path.join(
    DATASET_ROOT,
    "train"
)

VALIDATION_PATH = os.path.join(
    DATASET_ROOT,
    "validation"
)

OUTPUT_DIR = "./donut_sinhala_final"

MAX_LENGTH = 256

# Donut Swin requires dimensions compatible with its patch structure.
IMAGE_SIZE = [960, 640]


# ============================================================
# GPU CHECK
# ============================================================

print("=" * 70)
print("GPU / PYTORCH INFORMATION")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable a T4 GPU in Kaggle."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Compute capability:",
    torch.cuda.get_device_capability(0)
)

print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

# Basic CUDA test
x = torch.randn(
    256,
    256,
    device="cuda"
)

y = torch.randn(
    256,
    256,
    device="cuda"
)

z = x @ y

print("Basic CUDA test: PASSED")

del x, y, z
torch.cuda.empty_cache()


# ============================================================
# HUGGING FACE AUTHENTICATION
# ============================================================

try:

    HF_TOKEN = UserSecretsClient().get_secret(
        "HF_TOKEN"
    )

    os.environ["HF_TOKEN"] = HF_TOKEN

    print("\nHugging Face authentication: configured")

except Exception as e:

    HF_TOKEN = None

    print(
        "\nWarning: HF_TOKEN could not be loaded:",
        e
    )


if HF_TOKEN:

    try:

        user = whoami(token=HF_TOKEN)

        username = (
            user.get("name")
            or user.get("fullname")
            or "unknown"
        )

        print(
            "Hugging Face user:",
            username
        )

    except Exception as e:

        print(
            "HF authentication check failed:",
            e
        )


# ============================================================
# LOAD DONUT PROCESSOR
# ============================================================

print("\nLoading Donut processor...")

processor = DonutProcessor.from_pretrained(
    "naver-clova-ix/donut-base",
    use_fast=False
)

print("Donut processor loaded.")


# ============================================================
# LOAD SINBERT TOKENIZER
# ============================================================

print("\nLoading SinBERT tokenizer...")

sinhala_tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    use_fast=False,
    token=HF_TOKEN
)

print("SinBERT tokenizer loaded.")

# Replace Donut's tokenizer with SinBERT tokenizer
processor.tokenizer = sinhala_tokenizer


# ============================================================
# LOAD DONUT MODEL
# ============================================================

print("\nLoading Donut model...")

donut_model = VisionEncoderDecoderModel.from_pretrained(
    "naver-clova-ix/donut-base"
)

donut_encoder = donut_model.encoder

print("Donut Swin encoder loaded.")


# ============================================================
# LOAD CUSTOM TROCR MODEL
# ============================================================

print(
    "\nLoading custom TrOCR model:",
    TROCR_HUB_ID
)

trocr_model = VisionEncoderDecoderModel.from_pretrained(
    TROCR_HUB_ID,
    token=HF_TOKEN
)

trocr_decoder = trocr_model.decoder

print("Custom TrOCR decoder loaded.")


# ============================================================
# GET ENCODER / DECODER DIMENSIONS
# ============================================================

encoder_hidden_size = (
    donut_encoder.config.hidden_size
)

decoder_hidden_size = (
    trocr_decoder.config.hidden_size
)

print("\nArchitecture:")
print(
    "Donut encoder hidden size:",
    encoder_hidden_size
)

print(
    "TrOCR decoder hidden size:",
    decoder_hidden_size
)


# ============================================================
# CREATE CORRECT HYBRID CONFIG
# ============================================================

hybrid_config = (
    VisionEncoderDecoderConfig
    .from_encoder_decoder_configs(
        donut_encoder.config,
        trocr_decoder.config
    )
)


# ============================================================
# CREATE HYBRID MODEL
# ============================================================

model = VisionEncoderDecoderModel(
    config=hybrid_config,
    encoder=donut_encoder,
    decoder=trocr_decoder
)


# ============================================================
# ENCODER -> DECODER PROJECTION
# ============================================================

if encoder_hidden_size != decoder_hidden_size:

    print(
        "\nCreating encoder -> decoder projection:",
        encoder_hidden_size,
        "->",
        decoder_hidden_size
    )

    model.enc_to_dec_proj = torch.nn.Linear(
        encoder_hidden_size,
        decoder_hidden_size
    )

else:

    print(
        "\nEncoder and decoder dimensions match."
    )


# ============================================================
# MODEL CONFIGURATION
# ============================================================

model.config.tie_word_embeddings = False

model.config.pad_token_id = (
    processor.tokenizer.pad_token_id
)

model.config.eos_token_id = (
    processor.tokenizer.eos_token_id
)

decoder_start_token = (
    processor.tokenizer.cls_token_id
)

if decoder_start_token is None:

    decoder_start_token = (
        processor.tokenizer.bos_token_id
    )

if decoder_start_token is None:

    raise ValueError(
        "SinBERT tokenizer has neither "
        "cls_token_id nor bos_token_id."
    )

model.config.decoder_start_token_id = (
    decoder_start_token
)


# ============================================================
# IMAGE SIZE
# ============================================================

model.config.encoder.image_size = IMAGE_SIZE

processor.image_processor.size = {
    "height": IMAGE_SIZE[0],
    "width": IMAGE_SIZE[1]
}


# ============================================================
# ADD DONUT STRUCTURAL TOKEN
# ============================================================

print("\nAdding <s_gt_parse> token...")

num_added = processor.tokenizer.add_tokens(
    ["<s_gt_parse>"]
)

print(
    "Number of new tokens:",
    num_added
)

if num_added > 0:

    model.decoder.resize_token_embeddings(
        len(processor.tokenizer)
    )


model.config.vocab_size = (
    len(processor.tokenizer)
)


# ============================================================
# GRADIENT CHECKPOINTING
# ============================================================

model.gradient_checkpointing_enable()

# Required when using gradient checkpointing with caching.
model.config.use_cache = False


# ============================================================
# CLEAN TEMPORARY MODELS
# ============================================================

del donut_model
del trocr_model

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model = model.cuda()


# ============================================================
# FINAL MODEL INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("HYBRID MODEL READY")
print("=" * 70)

print("Encoder: Donut Swin")
print("Decoder: Custom Sinhala TrOCR")
print("Tokenizer: SinBERT")
print(
    "Encoder -> Decoder projection:",
    hasattr(model, "enc_to_dec_proj")
)

print(
    "Tokenizer vocabulary:",
    len(processor.tokenizer)
)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Model device:",
    next(model.parameters()).device
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 2 — DATASET + PRE-TRAINING TEST
# ============================================================

import os
import json
from PIL import Image
from torch.utils.data import Dataset as TorchDataset

# ============================================================
# DATASET PATH CHECK
# ============================================================

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Dataset root:", DATASET_ROOT)
print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)

assert os.path.exists(
    TRAIN_PATH
), f"Train path not found: {TRAIN_PATH}"

assert os.path.exists(
    VALIDATION_PATH
), f"Validation path not found: {VALIDATION_PATH}"


# ============================================================
# LOAD TRAIN & VALIDATION DATASETS PURELY AS LISTS
# ============================================================

def load_custom_donut_lists(split_path):
    jsonl_path = os.path.join(split_path, "metadata.jsonl")
    
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(f"metadata.jsonl not found at: {jsonl_path}")
    
    image_paths = []
    ground_truths = []
    
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            file_name = data["file_name"]
            ground_truth = data["ground_truth"]
            
            full_img_path = os.path.join(split_path, file_name)
            image_paths.append(full_img_path)
            ground_truths.append(ground_truth)
            
    return image_paths, ground_truths

print("Loading train lists...")
train_img_paths, train_gts = load_custom_donut_lists(TRAIN_PATH)

print("Loading validation lists...")
val_img_paths, val_gts = load_custom_donut_lists(VALIDATION_PATH)


# ============================================================
# PURE PYTORCH DATASET (Bypassing HF Arrow Batcher Conflicts)
# ============================================================

class DonutPyTorchDataset(TorchDataset):

    def __init__(
        self,
        image_paths,
        ground_truths,
        processor,
        max_length
    ):
        self.image_paths = image_paths
        self.ground_truths = ground_truths
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image via PIL
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        # Ground truth
        ground_truth = self.ground_truths[idx]

        if isinstance(ground_truth, str):
            try:
                ground_truth_json = json.loads(ground_truth)
            except json.JSONDecodeError:
                ground_truth_json = {"gt_parse": {}}
        else:
            ground_truth_json = ground_truth

        # Extract gt_parse
        if "gt_parse" not in ground_truth_json:
            if isinstance(ground_truth_json, dict) and len(ground_truth_json) > 0:
                gt_parse = ground_truth_json
            else:
                raise ValueError(
                    f"ground_truth at index {idx} does not contain 'gt_parse'."
                )
        else:
            gt_parse = ground_truth_json["gt_parse"]

        # Create target sequence
        target_sequence = (
            "<s_gt_parse>"
            + json.dumps(
                gt_parse,
                ensure_ascii=False
            )
            + self.processor.tokenizer.eos_token
        )

        # Process image
        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Tokenize target
        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )["input_ids"].squeeze(0)

        # Ignore padding during loss
        labels[
            labels
            == self.processor.tokenizer.pad_token_id
        ] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = DonutPyTorchDataset(
    train_img_paths,
    train_gts,
    processor,
    MAX_LENGTH
)

validation_dataset = DonutPyTorchDataset(
    val_img_paths,
    val_gts,
    processor,
    MAX_LENGTH
)

print("\nPyTorch datasets created successfully:")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))


# ============================================================
# INSPECT ONE SAMPLE
# ============================================================

sample = train_dataset[0]

print("\nSample:")
print(
    "pixel_values:",
    sample["pixel_values"].shape,
    sample["pixel_values"].dtype
)

print(
    "labels:",
    sample["labels"].shape,
    sample["labels"].dtype
)


# ============================================================
# DECODE SAMPLE TARGET
# ============================================================

display_labels = sample["labels"].clone()

display_labels[
    display_labels == -100
] = processor.tokenizer.pad_token_id

decoded = processor.tokenizer.decode(
    display_labels,
    skip_special_tokens=False
)

print("\nDecoded target:")
print(decoded[:1000])


# ============================================================
# PRE-TRAINING FORWARD TEST
# ============================================================

print("\n" + "=" * 70)
print("RUNNING PRE-TRAINING FORWARD/BACKWARD TEST")
print("=" * 70)

model.train()

pixel_values = (
    sample["pixel_values"]
    .unsqueeze(0)
    .cuda()
)

labels = (
    sample["labels"]
    .unsqueeze(0)
    .cuda()
)

# Forward pass
outputs = model(
    pixel_values=pixel_values,
    labels=labels
)

loss = outputs.loss

print(
    "Forward pass successful."
)

print(
    "Loss:",
    loss.item()
)

# Backward pass
loss.backward()

print(
    "Backward pass successful."
)

# Clean test tensors
del pixel_values
del labels
del outputs
del loss

torch.cuda.empty_cache()

print("=" * 70)
print("PRE-TRAINING TEST PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — TRAINING
# ============================================================

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)


# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = Seq2SeqTrainingArguments(

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    output_dir="./donut_sinhala_output",

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    num_train_epochs=10,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-5,

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    eval_strategy="epoch",

    per_device_eval_batch_size=1,

    # --------------------------------------------------------
    # Generation
    # --------------------------------------------------------

    predict_with_generate=True,

    # --------------------------------------------------------
    # Mixed precision
    # T4 supports FP16.
    # --------------------------------------------------------

    fp16=True,

    # --------------------------------------------------------
    # Memory
    # --------------------------------------------------------

    gradient_checkpointing=True,

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------

    save_strategy="epoch",

    save_total_limit=2,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_strategy="steps",

    logging_steps=1,

    report_to="none",

    # --------------------------------------------------------
    # Data loader
    # --------------------------------------------------------

    dataloader_num_workers=2,

    # --------------------------------------------------------
    # Important for custom model
    # --------------------------------------------------------

    remove_unused_columns=False,
)


# ============================================================
# CREATE TRAINER
# ============================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=processor,

)


# ============================================================
# START TRAINING
# ============================================================

print("\n" + "=" * 70)
print("STARTING SINHALA HANDWRITING TRAINING")
print("=" * 70)

print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(validation_dataset)
)

print(
    "Epochs:",
    training_args.num_train_epochs
)

print(
    "Batch size:",
    training_args.per_device_train_batch_size
)

print(
    "Gradient accumulation:",
    training_args.gradient_accumulation_steps
)

print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)


# ============================================================
# TRAIN
# ============================================================

train_result = trainer.train()


# ============================================================
# SAVE TRAINED MODEL
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)

FINAL_MODEL_PATH = (
    "./donut_sinhala_final"
)

trainer.save_model(
    FINAL_MODEL_PATH
)

processor.save_pretrained(
    FINAL_MODEL_PATH
)


# ============================================================
# SAVE TRAINING METRICS
# ============================================================

metrics = train_result.metrics

print("\nTraining metrics:")

for key, value in metrics.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# FINAL GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )

    print("\nGPU memory:")
    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved: {reserved:.2f} GB"
    )


print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Final model saved to:",
    FINAL_MODEL_PATH
)

In [ ]:
# ============================================================
# CELL 5 — EVALUATION OF DONUT PREDICTIONS
# ============================================================

import re
from collections import Counter

print("=" * 70)
print("EVALUATING DONUT MODEL ON VALIDATION SET")
print("=" * 70)

model.eval()

# Let's generate predictions for the validation dataset
predictions = []
references = []

print("\nGenerating predictions for validation samples...")

for idx in range(len(validation_dataset)):
    sample = validation_dataset[idx]
    
    # Move pixel values to GPU and add batch dimension
    pixel_values = sample["pixel_values"].unsqueeze(0).cuda()
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=MAX_LENGTH,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            decoder_start_token_id=model.config.decoder_start_token_id,
        )
    
    # Decode prediction
    pred_text = processor.tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False
    )
    
    # Decode reference label
    ref_labels = sample["labels"].clone()
    ref_labels[ref_labels == -100] = processor.tokenizer.pad_token_id
    ref_text = processor.tokenizer.decode(
        ref_labels,
        skip_special_tokens=False
    )
    
    predictions.append(pred_text)
    references.append(ref_text)

print(f"Generated {len(predictions)} predictions successfully.")


# ============================================================
# PARSING & METRIC COMPUTATION HELPER
# ============================================================

def extract_gt_parse_dict(text):
    """Extracts the JSON key-value pairs from the generated token sequence."""
    try:
        # Find json content inside the sequence
        match = re.search(r"<s_gt_parse>(.*?)(?:</s>|<pad>|$)", text)
        if match:
            json_str = match.group(1)
            return json.loads(json_str)
    except Exception:
        pass
    return {}

total_keys = 0
matched_keys = 0
pred_key_count = 0
ref_key_count = 0

for pred, ref in zip(predictions, references):
    pred_dict = extract_gt_parse_dict(pred)
    ref_dict = extract_gt_parse_dict(ref)
    
    pred_key_count += len(pred_dict)
    ref_key_count += len(ref_dict)
    
    # Exact key-value matching score approximation for entity alignment
    for k, v in ref_dict.items():
        if k in pred_dict and str(pred_dict[k]).strip() == str(v).strip():
            matched_keys += 1
        total_keys += 1

# Compute Token/Entity Level Precision, Recall, and F1 approximations
if pred_key_count > 0:
    ser_precision = matched_keys / pred_key_count
else:
    ser_precision = 0.0

if ref_key_count > 0:
    ser_recall = matched_keys / ref_key_count
else:
    ser_recall = 0.0

if (ser_precision + ser_recall) > 0:
    ser_f1 = 2 * (ser_precision * ser_recall) / (ser_precision + ser_recall)
else:
    ser_f1 = 0.0


# ============================================================
# PRINT EVALUATION METRICS REPORT
# ============================================================

print("\n" + "=" * 50)
print("DONUT MODEL VALIDATION RESULTS (SIMULATED SER METRICS)")
print("=" * 50)
print(f"Semantic Entity Recognition (SER):")
print(f"  Precision : {ser_precision:.4f}")
print(f"  Recall    : {ser_recall:.4f}")
print(f"  F1-Score  : {ser_f1:.4f}")
print("-" * 50)
print(f"Note: Evaluated across {len(validation_dataset)} validation samples.")
print("=" * 50)

In [ ]:
import os
import re
import json
import torch
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import (
    VisionEncoderDecoderModel,
    DonutProcessor,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

# ==========================================
# 1. CONFIGURATION
# ==========================================
USE_CUSTOM_TROCR_DECODER = True 

from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    print("HF_TOKEN not found. Public models only.")
    HF_TOKEN = None

DATASET_PATH = (
    "/kaggle/input/datasets/"
    "danushamsc25/sinfunddonut/"
    "SinFundDonut/train"
)
MAX_LENGTH = 512
IMAGE_SIZE = [960, 640]

# ==========================================
# 2. MODEL & PROCESSOR SETUP
# ==========================================
print("Loading architecture...")

if not USE_CUSTOM_TROCR_DECODER:
    processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
    model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
else:
    donut_processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
    sinhala_tokenizer = AutoTokenizer.from_pretrained("NLPC-UOM/SinBERT-large", use_fast=False, token=HF_TOKEN)
    donut_processor.tokenizer = sinhala_tokenizer
    processor = donut_processor 

    model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
    model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.encoder.image_size = IMAGE_SIZE
processor.image_processor.size = {"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]}
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id if processor.tokenizer.cls_token_id else processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

# Disable cache during training to completely avoid DynamicCache collation errors in multi-GPU/DataParallel setups
model.config.use_cache = False

# ==========================================
# 3. SPECIAL TOKENS SETUP
# ==========================================
new_tokens = [
    "<s_gt_parse>", "</s_gt_parse>",
    "<s_form>", "</s_form>",
    "<s_label>", "</s_label>",
    "<s_text>", "</s_text>"
]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

def json2token(obj):
    if type(obj) == dict:
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        else:
            output = ""
            for k, v in obj.items():
                output += f"<s_{k}>" + json2token(v) + f"</s_{k}>"
            return output
    elif type(obj) == list:
        return "".join([json2token(item) for item in obj])
    else:
        return str(obj)

# ==========================================
# 4. DATASET PREPARATION
# ==========================================
raw_dataset = load_dataset("imagefolder", data_dir=DATASET_PATH, split="train")
print("Available columns:", raw_dataset.column_names)

class DonutDataset(Dataset):
    def __init__(self, dataset, processor, max_length):
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        
        gt_raw = item.get("ground_truth", item.get("text", "{}"))
        ground_truth_json = json.loads(gt_raw) if isinstance(gt_raw, str) else gt_raw
        
        xml_sequence = json2token(ground_truth_json)
        target_sequence = xml_sequence + self.processor.tokenizer.eos_token
        
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        
        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )["input_ids"].squeeze()
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_dataset = DonutDataset(raw_dataset, processor, MAX_LENGTH)

# ==========================================
# 5. TRAINING
# ==========================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./donut_sinhala_output",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    dataloader_drop_last=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting Training...")
trainer.train()

# ==========================================
# 6. EVALUATION & INFERENCE TEST
# ==========================================
print("Running Inference Test on a sample...")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

sample = train_dataset[0]
pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

outputs = model.generate(
    pixel_values,
    max_length=MAX_LENGTH,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    use_cache=True,
    bad_words_ids=[[processor.tokenizer.unk_token_id]],
    return_dict_in_generate=True,
)

sequence = processor.batch_decode(outputs.sequences)[0]
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
sequence = re.sub(r"^.*?" + "<s_gt_parse>", "<s_gt_parse>", sequence, 1)

try:
    predicted_json = processor.token2json(sequence)
    print("✅ PREDICTED DICTIONARY:")
    print(json.dumps(predicted_json, indent=2, ensure_ascii=False))
except Exception as e:
    print("❌ Failed to parse JSON. Model needs more training epochs.")
    print("Raw sequence output:", sequence)